# Mímir — validation CPU, CUDA et TPU sur Colab

Choisir CPU, GPU NVIDIA ou **TPU-v5e1 / TPU-v6e1** dans Colab. Exécuter les cellules dans l'ordre.
Le checkout courant est utilisé sans correctifs embarqués ni modification des sources CPU.
Les tests CUDA/TPU exigent une exécution native réelle ; un repli CPU ne valide pas l'accélérateur.
Le montage Drive et l'entraînement sur dataset sont optionnels, bornés et désactivés par défaut.

Détection automatique : laissez `BACKEND="auto"`. Le notebook sélectionne CUDA si un GPU NVIDIA est disponible, sinon TPU (v5e1/v6e1), sinon CPU. `"cpu"`, `"gpu"`/`"cuda"` ou `"tpu"` permettent de forcer le choix. Les dépendances, le build et les mesures utilisent ce même backend ; un accélérateur sélectionné mais non fonctionnel provoque un échec explicite.


Au début : `GPU_CUDA`, `GPU_VULKAN`, `GPU_OPENCL` activent indépendamment les runtimes **Mímir** (ignorés en CPU/TPU). Le benchmark réel laisse le planner choisir parmi ceux activés ; le microbenchmark mesure chaque runtime validé séparément : `mimir-cuda`, `mimir-vulkan`, `mimir-opencl`. Les concurrents ne sont mesurés qu’une fois par cas. Pour isoler Vulkan ou OpenCL, activer uniquement celui-ci. Les concurrents utilisent CUDA sur GPU. Les runtimes indisponibles (code 77) sont exclus et consignés dans le rapport ; `GPU_REQUIRE_ALL=True` rend leur absence bloquante. Au moins un runtime GPU doit être utilisable. Un échec numérique reste bloquant ; le pilote/ICD GPU doit être fourni par Colab, les outils installés ne le remplacent pas.

`GPU_VULKAN_REPAIR=True` tente de restaurer les composants NVIDIA Vulkan dans `/content/mimir-nvidia-vulkan/<version>` en extrayant le paquet officiel correspondant exactement au pilote hôte (~378 Mio pour 580.82.07). Le pilote noyau et les bibliothèques CUDA système restent en place. Seul un GPU NVIDIA énuméré par Vulkan valide cette étape ; les tests natifs Mímir restent obligatoires. Audit : `vulkan-setup.json` dans le dossier de journaux.


## Configuration et journaux
Adapter REPO/BRANCH et BACKEND. Le mode auto sonde les périphériques dans des processus séparés.

In [ ]:
import os, sys, json, shutil, subprocess, time, platform, hashlib, signal
from pathlib import Path
from datetime import datetime, timezone
REPO_URL = "https://github.com/brit45/mimir-framework.git"
BRANCH = "develop"
REPO = Path("/content/mimir-framework")
BACKEND = "auto"                    # auto | cpu | gpu/cuda | tpu
# Mímir GPU runtimes: independent switches (competitors retain CUDA).
GPU_CUDA = True
GPU_VULKAN = False
GPU_VULKAN_REPAIR = True  # private exact-version NVIDIA userspace, no kernel driver installation
GPU_OPENCL = False
GPU_REQUIRE_ALL = False  # True: an unavailable requested GPU runtime is fatal
TPU_TARGET = "auto"                 # auto | TPU-v5e1 | TPU-v6e1
JOBS = min(2, os.cpu_count() or 1)
THREADS = min(2, os.cpu_count() or 1)
LOG_DIR = Path("/content/mimir-colab-logs")
LOG_DIR.mkdir(parents=True, exist_ok=True)

def run(args, cwd=None, env=None, timeout=1800, capture=True, allowed_returncodes=()):
    command=list(map(str,args))
    log=LOG_DIR/(datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S-%f')+'.log')
    print('+', ' '.join(command), '\nJournal :',log,flush=True)
    # The child writes directly to a file: timeout remains effective even when
    # a compiler/driver stops producing stdout. Display the new log incrementally.
    with log.open('w') as output:
        p=subprocess.Popen(command,cwd=cwd,env=env,stdout=output,stderr=subprocess.STDOUT,text=True,start_new_session=True)
        started=time.monotonic();offset=0
        try:
            while True:
                with log.open(errors='replace') as reader:
                    reader.seek(offset);chunk=reader.read();offset=reader.tell()
                if chunk:print(chunk,end='',flush=True)
                if p.poll() is not None:break
                if time.monotonic()-started>timeout:
                    os.killpg(p.pid,signal.SIGKILL);p.wait();raise TimeoutError(f'Commande trop longue ; journal : {log}')
                time.sleep(.2)
        except BaseException:
            if p.poll() is None:os.killpg(p.pid,signal.SIGKILL);p.wait()
            raise
    text=log.read_text(errors='replace')
    if p.returncode and p.returncode not in allowed_returncodes:raise RuntimeError(f'Code {p.returncode} ; journal : {log}\n'+text[-12000:])
    return text

def choose_backend(request):
    # Probe hardware in children so the notebook never owns an accelerator.
    request=request.lower()
    if request=='gpu':request='cuda'
    if request not in {'auto','cpu','cuda','tpu'}:raise ValueError(request)
    if request!='auto':return request
    if shutil.which('nvidia-smi'):
        try:
            probe=subprocess.run(['nvidia-smi','--query-gpu=name,compute_cap','--format=csv,noheader'],
                                 capture_output=True,text=True,timeout=15)
            if probe.returncode==0 and probe.stdout.strip():return 'cuda'
            probe=subprocess.run(['nvidia-smi','-L'],capture_output=True,text=True,timeout=15)
            if probe.returncode==0 and 'GPU' in probe.stdout:return 'cuda'
        except (OSError,subprocess.TimeoutExpired):pass
    if any(os.environ.get(k) for k in ('COLAB_TPU_ADDR','TPU_NAME','TPU_WORKER_ID','TPU_ACCELERATOR_TYPE')) or any(
            Path(p).exists() for p in ('/dev/accel0','/dev/accel/accel0')):return 'tpu'
    try:
        probe=subprocess.run([sys.executable,'-c',
            "import jax; print('MIMIR_TPU_COUNT='+str(len(jax.devices('tpu'))))"],
            capture_output=True,text=True,timeout=60)
        if probe.returncode==0 and any(line.startswith('MIMIR_TPU_COUNT=') and
                int(line.split('=',1)[1])>0 for line in probe.stdout.splitlines()):return 'tpu'
    except (OSError,ValueError,subprocess.TimeoutExpired):pass
    return 'cpu'
REQUESTED_BACKEND=BACKEND
BACKEND=choose_backend(BACKEND)
MIMIR_GPU_RUNTIMES=[name for name,enabled in [('cuda',GPU_CUDA),('vulkan',GPU_VULKAN),('opencl',GPU_OPENCL)] if enabled] if BACKEND=='cuda' else []
if BACKEND=='cuda' and not MIMIR_GPU_RUNTIMES:raise ValueError('Activer au moins un runtime GPU Mímir')
MIMIR_MICRO_RUNTIME=MIMIR_GPU_RUNTIMES[0] if MIMIR_GPU_RUNTIMES else BACKEND
BUILD=REPO/('build-colab-'+BACKEND+('-'+'-'.join(MIMIR_GPU_RUNTIMES) if MIMIR_GPU_RUNTIMES else ''))
RUNTIME_ENV=dict(os.environ, OMP_NUM_THREADS=str(THREADS), OPENBLAS_NUM_THREADS=str(THREADS),
    MKL_NUM_THREADS=str(THREADS), TF_NUM_INTRAOP_THREADS=str(THREADS), TF_NUM_INTEROP_THREADS='1',
    XLA_PYTHON_CLIENT_PREALLOCATE='false', TF_FORCE_GPU_ALLOW_GROWTH='true',
    XLA_USE_BF16='0', XLA_DOWNCAST_BF16='0',
    MIMIR_TPU='1' if BACKEND=='tpu' else '0', MIMIR_TPU_TARGET=TPU_TARGET,
    MIMIR_TPU_PYTHON=sys.executable, MIMIR_DISABLE_FPGA='1', MIMIR_DISABLE_ROCM='1',
    MIMIR_DISABLE_CUDA='0' if 'cuda' in MIMIR_GPU_RUNTIMES else '1',
    MIMIR_DISABLE_VULKAN='0' if 'vulkan' in MIMIR_GPU_RUNTIMES else '1', MIMIR_DISABLE_OPENCL='0' if 'opencl' in MIMIR_GPU_RUNTIMES else '1', MIMIR_BENCH_RUNTIME=MIMIR_MICRO_RUNTIME, MIMIR_ALLOCATOR_LOG='0')
# Make an explicit CPU selection effective even on an accelerator VM.
if BACKEND=='cpu':
    RUNTIME_ENV.update(CUDA_VISIBLE_DEVICES='',JAX_PLATFORMS='cpu',PJRT_DEVICE='CPU')
elif BACKEND=='cuda':
    RUNTIME_ENV.update(JAX_PLATFORMS='cuda',PJRT_DEVICE='CUDA')
else:
    RUNTIME_ENV.update(CUDA_VISIBLE_DEVICES='',JAX_PLATFORMS='tpu',PJRT_DEVICE='TPU')
ENVIRONMENT=dict(requested_backend=REQUESTED_BACKEND,
                 selected_backend=BACKEND,mimir_gpu_runtimes=MIMIR_GPU_RUNTIMES,mimir_micro_runtime=MIMIR_MICRO_RUNTIME,python=sys.version,platform=platform.platform())
print('Environnement detecte :',json.dumps(ENVIRONMENT,ensure_ascii=False))
print('Backend :',BACKEND,'| cible TPU :',TPU_TARGET,'| threads :',THREADS)

## Dépendances et checkout
Par défaut, les sources actuelles sont embarquées dans ce notebook et vérifiées par SHA256. Elles sont extraites dans un dossier dédié ; un checkout existant est conservé. `SOURCE_MODE="checkout"` permet de choisir explicitement la branche distante.

In [ ]:
# CPU setup must not inherit runtime requests from an earlier GPU session.
if BACKEND=='cpu':
    MIMIR_GPU_RUNTIMES=[]
    MIMIR_MICRO_RUNTIME='cpu'
    RUNTIME_ENV.update(MIMIR_BENCH_RUNTIME='cpu',MIMIR_DISABLE_CPU='0',MIMIR_TPU='0',
        MIMIR_DISABLE_CUDA='1',MIMIR_DISABLE_ROCM='1',MIMIR_DISABLE_VULKAN='1',
        MIMIR_DISABLE_OPENCL='1',CUDA_VISIBLE_DEVICES='',JAX_PLATFORMS='cpu',PJRT_DEVICE='CPU')
    if 'ENVIRONMENT' in globals():
        ENVIRONMENT.update(selected_backend='cpu',mimir_gpu_runtimes=[],mimir_micro_runtime='cpu')
apt=['apt-get'] if os.geteuid()==0 else ['sudo','apt-get']
run(apt+['update','-qq'])
run(apt+['install','-y','-qq','build-essential','g++-12','cmake','ninja-build','git',
         'pkg-config','util-linux','python3-venv','liblua5.3-dev','liblz4-dev','libssl-dev','nlohmann-json3-dev'])
# SDK/tools only: the accelerator driver/ICD must be supplied by the host.
if 'vulkan' in MIMIR_GPU_RUNTIMES:
    run(apt+['install','-y','-qq','libvulkan1','libvulkan-dev','vulkan-tools','glslang-tools','spirv-tools','binutils','libegl1','libglx0','libx11-6','libx11-xcb1','libxcb1','libxext6','libdrm2','libgbm1'])
if 'opencl' in MIMIR_GPU_RUNTIMES:
    run(apt+['install','-y','-qq','ocl-icd-opencl-dev','opencl-headers','clinfo'])
# BEGIN MIMIR EMBEDDED SOURCES
import base64, gzip, io, tarfile, tempfile
SOURCE_MODE = "embedded"  # embedded | checkout (explicit opt-in to REPO/BRANCH)
SOURCE_SHA256 = '13329882b1d941f40e55addfd8843573e65a82308347ffb6c6bb25747117ea79'
SOURCE_PAYLOAD = '...'
if SOURCE_MODE == 'embedded':
    packed = base64.b64decode(SOURCE_PAYLOAD, validate=True)
    if hashlib.sha256(packed).hexdigest() != SOURCE_SHA256:
        raise RuntimeError('Instantané des sources corrompu : recharger le notebook')
    # A dedicated immutable source directory preserves any existing checkout.
    snapshot_root = REPO.parent / ('mimir-source-' + SOURCE_SHA256[:16])
    def verify_snapshot(root, manifest):
        return all((root / name).is_file() and
                   hashlib.sha256((root / name).read_bytes()).hexdigest() == digest
                   for name, digest in manifest['files'].items())
    with tarfile.open(fileobj=io.BytesIO(gzip.decompress(packed)), mode='r:') as archive:
        members = archive.getmembers()
        for member in members:
            path = Path(member.name)
            if path.is_absolute() or '..' in path.parts or not member.isfile():
                raise RuntimeError('Chemin interdit dans les sources : ' + member.name)
        manifest = json.load(archive.extractfile('colab-source-manifest.json'))
        names = [m.name for m in members]
        if len(names) != len(set(names)) or set(names) != set(manifest['files']) | {'colab-source-manifest.json'}:
            raise RuntimeError('Inventaire des sources invalide')
        if snapshot_root.exists():
            if not verify_snapshot(snapshot_root, manifest):
                raise RuntimeError('Instantané local modifié : supprimer ce dossier puis relancer : ' + str(snapshot_root))
        else:
            snapshot_root.parent.mkdir(parents=True, exist_ok=True)
            with tempfile.TemporaryDirectory(prefix='mimir-extract-', dir=snapshot_root.parent) as temporary:
                temporary = Path(temporary)
                for member in members:
                    target = temporary / member.name
                    target.parent.mkdir(parents=True, exist_ok=True)
                    target.write_bytes(archive.extractfile(member).read())
                    target.chmod(member.mode)
                if not verify_snapshot(temporary, manifest):
                    raise RuntimeError('Empreintes des sources invalides')
                temporary.rename(snapshot_root)
    REPO = snapshot_root
    COMMIT = manifest['commit']
    DIRTY = manifest['provenance'] + '; snapshot_sha256=' + SOURCE_SHA256
elif SOURCE_MODE == 'checkout':
    if not REPO.exists():
        run(['git', 'clone', '--branch', BRANCH, REPO_URL, REPO])
    COMMIT = run(['git', 'rev-parse', 'HEAD'], cwd=REPO).strip()
    DIRTY = run(['git', 'status', '--short'], cwd=REPO)
else:
    raise ValueError('SOURCE_MODE doit être embedded ou checkout')
BUILD = REPO / f'build-colab-{BACKEND}'
print('Sources :', REPO, '| mode :', SOURCE_MODE)
# END MIMIR EMBEDDED SOURCES




















# Restore NVIDIA Vulkan userspace privately, matching the loaded host driver.
VULKAN_SETUP=dict(status='not_requested')
if 'vulkan' in MIMIR_GPU_RUNTIMES and GPU_VULKAN_REPAIR:
    # BEGIN MIMIR VULKAN HELPER
    import types
    vulkan_helper_source = '"""Private, exact-host-version NVIDIA Vulkan userspace for Colab.\n\nNever run nvidia-installer, install a kernel module, or replace system CUDA libs.\nOnly the NVIDIA official self-extractor is run with --extract-only.\n"""\nimport hashlib\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport re\nimport shutil\nimport subprocess\nimport urllib.request\n\nGRAPHICS_PREFIXES=(\'libEGL_nvidia.so\',\'libGLX_nvidia.so\',\'libnvidia-eglcore.so\',\n                   \'libnvidia-glcore.so\',\'libnvidia-glsi.so\',\'libnvidia-gpucomp.so\',\n                   \'libnvidia-allocator.so\',\'libnvidia-tls.so\',\'libnvidia-glvkspirv.so\',\'libnvidia-vulkan-producer.so\')\n\n\ndef nvidia_device(output):\n    # Inspect physical-device blocks, rather than loader messages mentioning NVIDIA.\n    return any(re.search(r\'vendorID\\s*=\\s*(?:0x10de|4318)\\b\',block,re.I) and\n               re.search(r\'deviceType\\s*=\\s*PHYSICAL_DEVICE_TYPE_(?:DISCRETE|INTEGRATED)_GPU\',block)\n               for block in re.split(r\'GPU\\d+\\s*:\',output))\n\n\ndef probe(environment):\n    p=subprocess.run([\'vulkaninfo\',\'--summary\'],env=environment,capture_output=True,text=True,timeout=90)\n    return p.returncode==0 and nvidia_device(p.stdout),p.stdout+p.stderr\n\n\ndef stage_libraries(extracted,destination):\n    """Copy native x86_64 graphics ELF files and reconstruct their actual SONAMEs."""\n    destination.mkdir(parents=True,exist_ok=True)\n    copied=[]\n    for source in sorted(Path(extracted).rglob(\'*.so*\')):\n        if source.is_symlink() or not source.is_file() or not source.name.startswith(GRAPHICS_PREFIXES):continue\n        with source.open(\'rb\') as stream:header=stream.read(20)\n        if len(header)<20 or header[:5]!=b\'\\x7fELF\\x02\' or header[18:20]!=b\'\\x3e\\x00\':continue\n        target=destination/source.name\n        shutil.copy2(source,target)\n        dynamic=subprocess.run([\'readelf\',\'-d\',str(target)],capture_output=True,text=True,check=True)\n        soname=re.search(r\'\\(SONAME\\).*?\\[([^\\]]+)\\]\',dynamic.stdout)\n        if soname:\n            name=soname.group(1)\n            if Path(name).name!=name:raise RuntimeError(\'Invalid library SONAME\')\n            link=destination/name\n            if link!=target:\n                link.unlink(missing_ok=True);link.symlink_to(target.name)\n        copied.append(target.name)\n    if not any(name.startswith(\'libEGL_nvidia.so\') for name in copied):\n        raise RuntimeError(\'NVIDIA archive has no native libEGL_nvidia\')\n    return copied\n\n\ndef configure(environment,root,run):\n    """Return (child environment, audit report); callers keep availability policy."""\n    original=dict(environment)\n    report=dict(status=\'unavailable\',method=\'private NVIDIA graphics userspace\')\n    try:\n        ready,output=probe(original)\n        if ready:\n            report.update(status=\'host_ready\',probe=output)\n            return original,report\n        report[\'host_probe\']=output\n        if platform.machine()!=\'x86_64\':raise RuntimeError(\'Automatic NVIDIA extraction requires x86_64\')\n        version_output=run([\'nvidia-smi\',\'--query-gpu=driver_version\',\'--format=csv,noheader\'],timeout=30)\n        versions=set(version_output.strip().splitlines())\n        if len(versions)!=1:raise RuntimeError(\'Cannot determine a single NVIDIA host driver version\')\n        version=versions.pop().strip()\n        if not re.fullmatch(r\'\\d{3}\\.\\d{2}(?:\\.\\d{2})?\',version):raise RuntimeError(\'Invalid NVIDIA version: \'+version)\n        root=Path(root).resolve()/version;root.mkdir(parents=True,exist_ok=True)\n        archive=root/f\'NVIDIA-Linux-x86_64-{version}.run\'\n        url=f\'https://download.nvidia.com/XFree86/Linux-x86_64/{version}/{archive.name}\'\n        report.update(driver_version=version,url=url)\n        if not archive.is_file():\n            partial=archive.with_suffix(\'.partial\')\n            print(\'Téléchargement des composants NVIDIA officiels :\',url,flush=True)\n            try:\n                with urllib.request.urlopen(url,timeout=120) as response,partial.open(\'wb\') as target:\n                    shutil.copyfileobj(response,target)\n                partial.replace(archive)\n            finally:partial.unlink(missing_ok=True)\n        digest=hashlib.sha256()\n        with archive.open(\'rb\') as stream:\n            for chunk in iter(lambda:stream.read(1024*1024),b\'\'):digest.update(chunk)\n        report[\'archive_sha256\']=digest.hexdigest()\n        extracted=root/\'extracted\'\n        if not (extracted/\'.mimir-complete\').exists():\n            if extracted.exists():shutil.rmtree(extracted)\n            run([\'sh\',archive,\'--extract-only\',\'--target\',extracted],timeout=600)\n            (extracted/\'.mimir-complete\').touch()\n        library_dir=root/\'lib\'\n        report[\'libraries\']=stage_libraries(extracted,library_dir)\n        manifests=list(extracted.glob(\'nvidia_icd.json\'))\n        if not manifests:raise RuntimeError(\'Official NVIDIA ICD manifest absent from archive\')\n        icd=json.loads(manifests[0].read_text())\n        egl=next(library_dir/name for name in report[\'libraries\'] if name.startswith(\'libEGL_nvidia.so\'))\n        icd[\'ICD\'][\'library_path\']=str(egl)\n        manifest=root/\'nvidia_icd.json\';manifest.write_text(json.dumps(icd,indent=2))\n        candidate=dict(original,VK_DRIVER_FILES=str(manifest),VK_ICD_FILENAMES=str(manifest),\n                       LD_LIBRARY_PATH=str(library_dir)+(\':\'+original[\'LD_LIBRARY_PATH\'] if original.get(\'LD_LIBRARY_PATH\') else \'\'))\n        ready,output=probe(candidate)\n        report.update(probe=output,icd=str(manifest))\n        if not ready:raise RuntimeError(\'Private driver did not enumerate an NVIDIA Vulkan GPU; native tests remain authoritative\')\n        report[\'status\']=\'private_ready\'\n        return candidate,report\n    except Exception as exc:\n        report[\'error\']=str(exc)\n        print(\'Vulkan NVIDIA :\',str(exc),flush=True)\n        return original,report\n'
    vulkan_helper_sha256 = 'd3fe36f43d0a66be7ac6906f10feb9d423afc20378493658139658b5d10f5d23'
    if hashlib.sha256(vulkan_helper_source.encode()).hexdigest()!=vulkan_helper_sha256:raise RuntimeError('Module Vulkan embarque corrompu')
    vulkan_setup_module=types.ModuleType('mimir_nvidia_vulkan')
    exec(compile(vulkan_helper_source,'<mimir_nvidia_vulkan_embedded>','exec'),vulkan_setup_module.__dict__)
    # END MIMIR VULKAN HELPER
    RUNTIME_ENV,VULKAN_SETUP=vulkan_setup_module.configure(RUNTIME_ENV,Path('/content/mimir-nvidia-vulkan'),run)
ENVIRONMENT['vulkan_setup']=VULKAN_SETUP
(LOG_DIR/'vulkan-setup.json').write_text(json.dumps(VULKAN_SETUP,indent=2))
print('VULKAN_SETUP',VULKAN_SETUP.get('status'),VULKAN_SETUP.get('error',''))
# Vulkan loader/driver diagnostics: saved by run() even when no ICD is exposed.
if 'vulkan' in MIMIR_GPU_RUNTIMES:
    vulkan_probe = r"""
import glob,json,os,shutil,subprocess
paths=sorted(glob.glob('/usr/share/vulkan/icd.d/*.json')+glob.glob('/etc/vulkan/icd.d/*.json'))
print('VULKAN_ICD_FILES',json.dumps(paths),flush=True)
for path in paths:
    try:
        with open(path) as stream:print(path,stream.read(),flush=True)
    except OSError as exc:print(path,str(exc),flush=True)
commands=[['nvidia-smi'],['ldconfig','-p'],['vulkaninfo','--summary']]
for command in commands:
    print('DIAGNOSTIC',command,flush=True)
    if not shutil.which(command[0]):
        print('Commande absente',command[0],flush=True);continue
    env=dict(os.environ)
    if command[0]=='vulkaninfo':env['VK_LOADER_DEBUG']='all'
    try:
        p=subprocess.run(command,env=env,capture_output=True,text=True,timeout=60)
        output=p.stdout+p.stderr
        if command[0]=='ldconfig':output='\n'.join(line for line in output.splitlines() if any(name in line for name in ('libGLX_nvidia','libEGL_nvidia','libvulkan')))
        print(output,flush=True);print('DIAGNOSTIC_EXIT_CODE',p.returncode,flush=True)
    except (OSError,subprocess.TimeoutExpired) as exc:print(str(exc),flush=True)
"""
    run([sys.executable,'-c',vulkan_probe],env=RUNTIME_ENV,timeout=210)











required=['run_mimir.sh','src/runtimes/cpu/CpuRuntime.cpp','src/runtimes/RuntimeRouter.cpp',
          'src/runtimes/losses/LossDispatch.cpp','Tests/test_reconstruction_runtime.cpp']
if BACKEND=='tpu':required+=['src/runtimes/tpu/TpuRuntime.cpp','src/runtimes/tpu/worker.py',
                           'Tests/test_tpu_cpu_fixture.cpp','Tests/test_tpu_model_train.cpp']
if BACKEND=='cuda' and 'cuda' in MIMIR_GPU_RUNTIMES:required+=['src/runtimes/cuda/CudaNativeOps.cuh','Tests/test_cuda_native_ops.cpp']
missing=[p for p in required if not (REPO/p).is_file()]
if missing:raise RuntimeError('Sources incomplètes ; en mode checkout, utiliser embedded ou synchroniser la branche : '+str(missing))
CPU_HASHES={str(p.relative_to(REPO)):hashlib.sha256(p.read_bytes()).hexdigest()
            for p in (REPO/'src/runtimes/cpu').rglob('*') if p.is_file()}
if BACKEND=='tpu':run([sys.executable,'-m','pip','install','jax[tpu]'])
NVCC=shutil.which('nvcc') or ('/usr/local/cuda/bin/nvcc' if Path('/usr/local/cuda/bin/nvcc').is_file() else None)
if BACKEND=='cuda' and 'cuda' in MIMIR_GPU_RUNTIMES:
    if not NVCC:raise RuntimeError('Backend CUDA demandé, mais nvcc est absent')
    run(['nvidia-smi']);run([NVCC,'--version'])
BUILD.mkdir(parents=True,exist_ok=True)
CMAKE_ARGS=['cmake','-S',REPO,'-B',BUILD,'-G','Ninja','-DCMAKE_BUILD_TYPE=Release',
    '-DCMAKE_CXX_COMPILER=/usr/bin/g++-12','-DMIMIR_RUNTIME_OUTPUT_DIRECTORY='+str(BUILD/'bin'),
    '-DCMAKE_EXPORT_COMPILE_COMMANDS=ON','-DMIMIR_ENABLE_TESTS=ON','-DBUILD_TESTING=ON',
    '-DENABLE_CUDA='+('ON' if BACKEND=='cuda' and 'cuda' in MIMIR_GPU_RUNTIMES else 'OFF'),'-DENABLE_ROCM=OFF',
    '-DENABLE_VULKAN='+('ON' if BACKEND=='cuda' and 'vulkan' in MIMIR_GPU_RUNTIMES else 'OFF'),'-DENABLE_OPENCL='+('ON' if BACKEND=='cuda' and 'opencl' in MIMIR_GPU_RUNTIMES else 'OFF'),'-DMIMIR_AUTO_DETECT_GPU_BACKENDS=OFF',
    '-DMIMIR_VIZ_BACKEND=NONE','-DENABLE_SFML=OFF','-DENABLE_FFMPEG=OFF',
    '-DENABLE_SCRIPTING_RUST=OFF','-DENABLE_SCRIPTING_JS=OFF','-DENABLE_SCRIPTING_CSHARP=OFF',
    '-DMIMIR_PORTABLE_BUILD=ON']
if BACKEND=='cuda' and 'cuda' in MIMIR_GPU_RUNTIMES:CMAKE_ARGS+=['-DCMAKE_CUDA_HOST_COMPILER=/usr/bin/g++-12',f'-DCMAKE_CUDA_COMPILER={NVCC}']

## Compilation headless

In [ ]:
TEST_TARGETS=['RuntimeTest.MathLinear','RuntimeTest.MathConv2d','RuntimeTest.MathNorms',
              'RuntimeTest.MathAttention','RuntimeTest.NMS','AutogradTest.RuntimeDispatch','RuntimeTest.OptimizerDispatch','RuntimeTest.ReconstructionLoss','ModelTest.ReconLosses','ModelTest.ExtendedOptimizers']
# Core allocation, tiled matrix and attention regressions (CPU reference).
TEST_TARGETS += ['RuntimeTest.LinearBackwardTiled', 'RuntimeTest.TypedConversions', 'RuntimeTest.ConvolutionScratch', 'RuntimeTest.AllocatorScratch', 'RuntimeTest.MatMulBackward', 'RuntimeTest.BatchMatMulForward', 'RuntimeTest.FlatJoin', 'RuntimeTest.FiniteCheck', 'RuntimeTest.RouteRefresh', 'RuntimeTest.AttentionTiled']
TEST_TARGETS += ['RuntimeTest.OptimizerSelection']
if 'cuda' in MIMIR_GPU_RUNTIMES:TEST_TARGETS+=['RuntimeTest.CudaNativeOps']
if any(r in MIMIR_GPU_RUNTIMES for r in ('vulkan','opencl')):TEST_TARGETS+=['PortableNativeOps']
if BACKEND=='tpu':TEST_TARGETS+=['RuntimeTest.TpuCpuFixture','RuntimeTest.TpuModelTrain']

run(CMAKE_ARGS)
run(['cmake','--build',BUILD,'-j',JOBS,'--target','mimir',*TEST_TARGETS],timeout=2400)

## Validation numérique et matériel
Les six tests communs sont complétés par la parité CUDA native ou les tests matériels TPU et la boucle native Mímir.

In [ ]:
VALIDATED=False
# A resumed session may have run an older configuration cell.
GPU_REQUIRE_ALL = globals().get('GPU_REQUIRE_ALL', False)
# Clear GPU state when this cell is resumed after switching to CPU.
if BACKEND=='cpu':
    MIMIR_GPU_RUNTIMES=[]
    MIMIR_MICRO_RUNTIME='cpu'
    RUNTIME_ENV.update(MIMIR_BENCH_RUNTIME='cpu',MIMIR_DISABLE_CPU='0',MIMIR_TPU='0',
        MIMIR_DISABLE_CUDA='1',MIMIR_DISABLE_ROCM='1',MIMIR_DISABLE_VULKAN='1',
        MIMIR_DISABLE_OPENCL='1',CUDA_VISIBLE_DEVICES='',JAX_PLATFORMS='cpu',PJRT_DEVICE='CPU')
# These analytical tests also contain compiled-in GPU branches. Build the
# reference without GPU backends so it cannot activate them implicitly.
REFERENCE_TARGETS=['RuntimeTest.MathLinear','RuntimeTest.MathConv2d','RuntimeTest.MathNorms',
                   'RuntimeTest.MathAttention','RuntimeTest.NMS','AutogradTest.RuntimeDispatch','RuntimeTest.OptimizerDispatch','RuntimeTest.ReconstructionLoss','ModelTest.ReconLosses','ModelTest.ExtendedOptimizers']
REFERENCE_TARGETS += ['RuntimeTest.LinearBackwardTiled', 'RuntimeTest.TypedConversions', 'RuntimeTest.ConvolutionScratch', 'RuntimeTest.AllocatorScratch', 'RuntimeTest.MatMulBackward', 'RuntimeTest.BatchMatMulForward', 'RuntimeTest.FlatJoin', 'RuntimeTest.FiniteCheck', 'RuntimeTest.RouteRefresh', 'RuntimeTest.AttentionTiled']
REFERENCE_TARGETS += ['RuntimeTest.OptimizerSelection']
REFERENCE_BUILD=BUILD
REFERENCE_ENV=dict(RUNTIME_ENV,MIMIR_TPU='0',MIMIR_DISABLE_CPU='0',
    MIMIR_DISABLE_CUDA='1',MIMIR_DISABLE_ROCM='1',MIMIR_DISABLE_VULKAN='1',MIMIR_DISABLE_OPENCL='1')
# An existing GPU build must not become the CPU reference after a mode switch.
GPU_BUILD_OPTIONS=('-DENABLE_CUDA=','-DENABLE_ROCM=','-DENABLE_VULKAN=',
                   '-DENABLE_OPENCL=','-DMIMIR_AUTO_DETECT_GPU_BACKENDS=')
REFERENCE_HAS_GPU=any(str(arg).startswith(GPU_BUILD_OPTIONS) and
                      str(arg).split('=',1)[1].upper() in {'ON','TRUE','YES','1'} for arg in CMAKE_ARGS)
if BACKEND!='cpu' or REFERENCE_HAS_GPU:
    REFERENCE_BUILD=REPO/'build-colab-reference-cpu'
    reference_args=[]
    for arg in CMAKE_ARGS:
        value=str(arg)
        if value.startswith(('-DCMAKE_CUDA_','-DCMAKE_PROJECT_INCLUDE=')):continue
        if value.startswith(GPU_BUILD_OPTIONS):value=value.split('=',1)[0]+'=OFF'
        if value.startswith('-DMIMIR_RUNTIME_OUTPUT_DIRECTORY='):value='-DMIMIR_RUNTIME_OUTPUT_DIRECTORY='+str(REFERENCE_BUILD/'bin')
        reference_args.append(value)
    reference_args[reference_args.index('-B')+1]=str(REFERENCE_BUILD)
    run(reference_args+['-DCMAKE_PROJECT_INCLUDE='],timeout=600)
    run(['cmake','--build',REFERENCE_BUILD,'-j',JOBS,'--target',*REFERENCE_TARGETS],timeout=2400)
run(['ctest','--test-dir',REFERENCE_BUILD,'--output-on-failure','--no-tests=error','-R',
     r'^(RuntimeTest\.(MathLinear|MathConv2d|MathNorms|MathAttention|NMS|OptimizerDispatch|OptimizerSelection|ReconstructionLoss|LinearBackwardTiled|TypedConversions|ConvolutionScratch|AllocatorScratch|MatMulBackward|BatchMatMulForward|FlatJoin|FiniteCheck|RouteRefresh|AttentionTiled)|AutogradTest.RuntimeDispatch|ModelTest.ReconLosses|ModelTest.ExtendedOptimizers)$'],
    cwd=REPO,env=REFERENCE_ENV,timeout=600)
if 'cuda' in MIMIR_GPU_RUNTIMES:
    cuda_validation_env=dict(REFERENCE_ENV,MIMIR_DISABLE_CUDA='0')
    output=run([BUILD/'Tests/RuntimeTest.CudaNativeOps'],cwd=REPO,env=cuda_validation_env,timeout=600)
    if 'CUDA_NATIVE_PARITY_OK' not in output or 'SKIP:' in output:raise RuntimeError('CUDA natif non validé')
    output=run([BUILD/'Tests/RuntimeTest.OptimizerDispatch','CUDA'],cwd=REPO,env=cuda_validation_env,timeout=600)
    if 'OPTIMIZER_RUNTIME_OK CUDA' not in output:raise RuntimeError('Optimizers CUDA non valides')
    output=run([BUILD/'Tests/RuntimeTest.ReconstructionLoss','CUDA'],cwd=REPO,env=cuda_validation_env,timeout=600)
    if 'RECONSTRUCTION_RUNTIME_OK CUDA' not in output:raise RuntimeError('Loss de reconstruction CUDA non valides')
# Compiled support and usable host drivers are separate capabilities.
REQUESTED_GPU_RUNTIMES=list(MIMIR_GPU_RUNTIMES)
UNAVAILABLE_GPU_RUNTIMES={}
ACTIVE_GPU_RUNTIMES=[]
for runtime in REQUESTED_GPU_RUNTIMES:
    if runtime=='cuda':
        ACTIVE_GPU_RUNTIMES.append(runtime)  # CUDA parity passed above.
        continue
    runtime_validation_env=dict(REFERENCE_ENV)
    runtime_validation_env['MIMIR_DISABLE_'+runtime.upper()]='0'
    output=run([BUILD/'Tests/PortableNativeOps',runtime.upper()],cwd=REPO,env=runtime_validation_env,
               timeout=600,allowed_returncodes=(77,))
    if 'SKIP:' in output:
        UNAVAILABLE_GPU_RUNTIMES[runtime]=output[-12000:]
        RUNTIME_ENV['MIMIR_DISABLE_'+runtime.upper()]='1'
        print(runtime.upper(),'indisponible sur cet hôte ; exclu des mesures GPU')
        if globals().get('GPU_REQUIRE_ALL', False):raise RuntimeError(runtime+' demandé mais indisponible ; GPU_REQUIRE_ALL=True')
        continue
    if 'GPU_NATIVE_PARITY_OK' not in output:raise RuntimeError(runtime+' natif non validé')
    output=run([BUILD/'Tests/RuntimeTest.OptimizerDispatch',runtime.upper()],cwd=REPO,env=runtime_validation_env,timeout=600,allowed_returncodes=(77,))
    if 'SKIP:' in output:
        UNAVAILABLE_GPU_RUNTIMES[runtime]='optimizer: '+output[-12000:]
        RUNTIME_ENV['MIMIR_DISABLE_'+runtime.upper()]='1'
        print(runtime.upper(),'optimizer GPU indisponible ; runtime exclu des mesures')
        if globals().get('GPU_REQUIRE_ALL', False):raise RuntimeError(runtime+' optimizer GPU indisponible ; GPU_REQUIRE_ALL=True')
        continue
    if 'OPTIMIZER_RUNTIME_OK '+runtime.upper() not in output:raise RuntimeError('Optimizers '+runtime+' non valides')
    output=run([BUILD/'Tests/RuntimeTest.ReconstructionLoss',runtime.upper()],cwd=REPO,env=runtime_validation_env,timeout=600,allowed_returncodes=(77,))
    if 'SKIP:' in output:
        UNAVAILABLE_GPU_RUNTIMES[runtime]='reconstruction_loss: '+output[-12000:]
        RUNTIME_ENV['MIMIR_DISABLE_'+runtime.upper()]='1'
        if globals().get('GPU_REQUIRE_ALL', False):raise RuntimeError(runtime+' loss GPU indisponible ; GPU_REQUIRE_ALL=True')
        continue
    if 'RECONSTRUCTION_RUNTIME_OK '+runtime.upper() not in output:raise RuntimeError('Loss de reconstruction '+runtime+' non valides')
    ACTIVE_GPU_RUNTIMES.append(runtime)
if BACKEND=='cuda':
    if not ACTIVE_GPU_RUNTIMES:raise RuntimeError('Aucun runtime GPU Mímir utilisable ; activer CUDA ou fournir un pilote/ICD Vulkan/OpenCL. Aucun repli CPU implicite.')
    MIMIR_GPU_RUNTIMES=ACTIVE_GPU_RUNTIMES
    MIMIR_MICRO_RUNTIME=ACTIVE_GPU_RUNTIMES[0]
    RUNTIME_ENV['MIMIR_BENCH_RUNTIME']=MIMIR_MICRO_RUNTIME
ENVIRONMENT.update(mimir_micro_runtimes=list(MIMIR_GPU_RUNTIMES) if BACKEND=='cuda' else [BACKEND],
                   mimir_gpu_runtimes_requested=REQUESTED_GPU_RUNTIMES,
                   mimir_gpu_runtimes=list(MIMIR_GPU_RUNTIMES),mimir_micro_runtime=MIMIR_MICRO_RUNTIME,
                   unavailable_gpu_runtimes=UNAVAILABLE_GPU_RUNTIMES)
if BACKEND=='tpu':
    output=run([BUILD/'Tests/RuntimeTest.TpuCpuFixture','--tpu'],cwd=REPO,env=RUNTIME_ENV,timeout=1800)
    if 'TPU_HARDWARE_PARITY_OK' not in output:raise RuntimeError('Parité matérielle TPU non validée')
    output=run([BUILD/'Tests/RuntimeTest.TpuModelTrain'],cwd=REPO,env=RUNTIME_ENV,timeout=600)
    if 'TPU_MODEL_TRAIN_OK' not in output:raise RuntimeError('Entraînement natif Mímir/TPU non validé')
    output=run([BUILD/'Tests/RuntimeTest.OptimizerDispatch','TPU'],cwd=REPO,env=RUNTIME_ENV,timeout=1800)
    if 'OPTIMIZER_RUNTIME_OK TPU' not in output:raise RuntimeError('Optimizers TPU non valides')
    output=run([BUILD/'Tests/RuntimeTest.ReconstructionLoss','TPU'],cwd=REPO,env=RUNTIME_ENV,timeout=1800)
    if 'RECONSTRUCTION_RUNTIME_OK TPU' not in output:raise RuntimeError('Loss de reconstruction TPU non valides')
assert CPU_HASHES=={str(p.relative_to(REPO)):hashlib.sha256(p.read_bytes()).hexdigest()
                   for p in (REPO/'src/runtimes/cpu').rglob('*') if p.is_file()}, 'Sources CPU modifiées'
VALIDATED=True
print('VALIDATION_OK',BACKEND,COMMIT)

## Entraînement VAEConv optionnel
Le profil 64×64 sert à valider le pipeline ; il ne constitue pas une preuve de qualité sur des images haute résolution.

In [ ]:
# Explicit opt-in; fresh architecture uses resume=0 and a separate output folder.
RUN_TRAINING=False
DATASET=Path('/content/drive/MyDrive/prepared_lora_dataset')
OUT_DIR=Path('/content/mimir-vae-validation')
IMAGE_SIDE=64
EPOCHS=1
if RUN_TRAINING:
    if not VALIDATED:raise RuntimeError('Valider le backend avant de lancer un entraînement')
    if not DATASET.is_dir():
        from google.colab import drive
        drive.mount('/content/drive')
    if not DATASET.is_dir():raise FileNotFoundError(f'Dataset absent : {DATASET}')
    # Native filesystem probe detects Drive mounts invisible to the C++ loader.
    probe_source=BUILD/'dataset-path.cpp';probe_binary=BUILD/'dataset-path'
    probe_source.write_text('#include <filesystem>\nint main(int n,char**v){return n==2 && std::filesystem::is_directory(v[1])?0:1;}\n')
    run(['/usr/bin/g++-12','-std=c++17',probe_source,'-o',probe_binary])
    p=subprocess.run([str(probe_binary),str(DATASET)])
    if p.returncode:
        local=Path('/content/mimir-datasets')/DATASET.name
        local.parent.mkdir(parents=True,exist_ok=True)
        required=sum(p.stat().st_size for p in DATASET.rglob('*') if p.is_file())
        if required+2*1024**3>shutil.disk_usage(local.parent).free:raise RuntimeError('Espace insuffisant pour copier le dataset Drive en local')
        shutil.copytree(DATASET,local,dirs_exist_ok=True,symlinks=False)
        run([probe_binary,local]);DATASET=local
    if OUT_DIR.exists() and any(OUT_DIR.iterdir()):raise RuntimeError('Choisir un nouveau OUT_DIR pour cet entraînement frais')
    OUT_DIR.mkdir(parents=True,exist_ok=True)
    training_env=dict(RUNTIME_ENV,MIMIR_EXECUTABLE=str((BUILD/'bin/mimir').resolve()))
    run(['bash',REPO/'run_mimir.sh','--lua',REPO/'scripts/training/train_vae_conv.lua','--',
         '--dataset-root',DATASET,'--image-h',IMAGE_SIDE,'--image-w',IMAGE_SIDE,
         '--latent-h',8,'--latent-w',8,'--latent-c',16,'--base-channels',8,
         '--epochs',EPOCHS,'--resume',0,'--out-dir',OUT_DIR,
         '--csv-path',OUT_DIR/'history-loss.csv'],cwd=REPO,env=training_env,timeout=3600)
else:print('Entraînement dataset désactivé ; passer RUN_TRAINING=True pour le lancer.')

Les journaux restent dans `/content/mimir-colab-logs`. Les sorties de `/content` sont temporaires : les copier vers Drive si nécessaire.
Pour une comparaison mesurée avec PyTorch, TensorFlow et JAX, utiliser **Colab-bench-multi-framework.ipynb**.